# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakshmitejaswiniparuchuri/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Heuristic Rule & Reason Codes (Content Refresh Lane)

#### The Rule in Plain Words
Starting from eligible pages with meaningful baseline visibility (`impressions_prev_30d >= 100`), we compute an interpretable **Baseline Action Score** (0 to 100) using only features knowable at the decision moment:
1. **Staleness Score (up to 40 pts):** Pages that have gone longer without an update (`days_since_last_update`) receive higher urgency, capped at 365+ days.
2. **Striking-Distance Rank Bonus (up to 35 pts):** Pages ranking in positions `4` through `20` (`avg_position`) receive higher priority because small content refreshes on page-one/top-page-two results capture far more incremental clicks than buried pages (`position > 30`).
3. **Visibility Weight (up to 25 pts):** Pages with higher prior-month search exposure (`log1p(impressions_prev_30d)`) score higher so editorial effort targets pages with real audience demand.

Every scored page is assigned a primary **Reason Code** explaining why it was surfaced so editors know *what* to inspect first.

#### Reason Codes Output by the Rule
* **`STALE_STRIKING_DISTANCE`:** Page ranks in striking distance (`avg_position` between `4` and `20`) **and** has not been updated in over `180` days (`days_since_last_update > 180`). *Recommended action: Full content freshness update & intent check.*
* **`HIGH_VISIBILITY_AGING`:** Page has high prior visibility (`impressions_prev_30d >= 1000`) and moderate-to-high age (`days_since_last_update > 120`), even if outside positions 4–20. *Recommended action: Audit SERP competitiveness and refresh outdated sections.*
* **`STRIKING_DISTANCE_OPTIMIZE`:** Page ranks in positions `4`–`20` with relatively fresh content (`days_since_last_update <= 180`). *Recommended action: Optimize title/meta snippet and subheadings to lift CTR.*
* **`LONG_TAIL_STALE`:** Page is older (`days_since_last_update > 180`) with lower visibility or deeper rank (`avg_position > 20`). *Recommended action: Consolidate or lightly update only if topic remains strategically relevant.*

In [2]:
import os
import numpy as np
import pandas as pd

# Load starter dataset directly from GitHub (no HF token needed)
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
try:
    df = pd.read_csv(url)
except Exception:
    df = pd.read_csv("../data/raw/content_refresh_anonymized.csv")

# Filter to eligible slice (knowable at decision moment)
eligible = df[df["impressions_prev_30d"].notna() & (df["impressions_prev_30d"] >= 100)].copy()

def compute_baseline_score(frame: pd.DataFrame) -> pd.Series:
    # 1. Staleness component (0-40 pts, capped at 365 days)
    stale_days = frame["days_since_last_update"].fillna(frame["content_age_days"]).fillna(0).clip(0, 365)
    staleness_pts = (stale_days / 365.0) * 40.0

    # 2. Striking-distance position component (0-35 pts; peak at positions 4-20)
    pos = frame["avg_position"].fillna(50)
    pos_pts = np.where(
        (pos >= 4) & (pos <= 20),
        35.0 * (1.0 - (pos - 4.0) / 20.0),
        np.where(pos < 4, 15.0, 5.0)
    )

    # 3. Prior visibility component (0-25 pts, log-scaled)
    log_imp = np.log1p(frame["impressions_prev_30d"].clip(lower=0))
    vis_pts = (log_imp / log_imp.max()) * 25.0

    return (staleness_pts + pos_pts + vis_pts).round(2)

def assign_reason_code(row: pd.Series) -> str:
    stale = row["days_since_last_update"] if pd.notna(row["days_since_last_update"]) else 0
    pos = row["avg_position"] if pd.notna(row["avg_position"]) else 50
    imp = row["impressions_prev_30d"]

    if (4 <= pos <= 20) and (stale > 180):
        return "STALE_STRIKING_DISTANCE"
    elif (imp >= 1000) and (stale > 120):
        return "HIGH_VISIBILITY_AGING"
    elif (4 <= pos <= 20):
        return "STRIKING_DISTANCE_OPTIMIZE"
    else:
        return "LONG_TAIL_STALE"

print(f"Eligible pages ready for scoring: {len(eligible):,}")

Eligible pages ready for scoring: 18,010


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Building & Exporting the Ranked Editorial Queue

We score all eligible pages, sort them in descending order of `baseline_action_score` (breaking ties by `impressions_prev_30d`), assign an integer `priority_rank` (`1` = highest priority), and export the ranked queue to `work/outputs/baseline_action_score.csv`.

In [3]:
eligible["baseline_action_score"] = compute_baseline_score(eligible)
eligible["reason_code"] = eligible.apply(assign_reason_code, axis=1)

# Sort highest score first, then highest prior impressions
ranked_queue = eligible.sort_values(
    by=["baseline_action_score", "impressions_prev_30d"],
    ascending=[False, False]
).reset_index(drop=True)

ranked_queue["priority_rank"] = ranked_queue.index + 1

# Ensure output directory exists and save CSV
os.makedirs("../outputs", exist_ok=True)
os.makedirs("work/outputs", exist_ok=True)

output_cols = [
    "priority_rank",
    "content_id",
    "client_id",
    "baseline_action_score",
    "reason_code",
    "days_since_last_update",
    "avg_position",
    "impressions_prev_30d",
    "word_count",
]

# Write to work/outputs/baseline_action_score.csv (and ../outputs if running inside work/notebooks/)
out_path = "work/outputs/baseline_action_score.csv"
ranked_queue[output_cols].to_csv(out_path, index=False)
ranked_queue[output_cols].to_csv("../outputs/baseline_action_score.csv", index=False)

print(f"Saved {len(ranked_queue):,} ranked pages to {out_path}")
print("\nReason code distribution across entire queue:")
print(ranked_queue["reason_code"].value_counts())


Saved 18,010 ranked pages to work/outputs/baseline_action_score.csv

Reason code distribution across entire queue:
reason_code
STRIKING_DISTANCE_OPTIMIZE    11668
LONG_TAIL_STALE                6322
STALE_STRIKING_DISTANCE          14
HIGH_VISIBILITY_AGING             6
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Editorial Queue Review

The code cell below generates a per-row audit table for **Ranks #1 through #20**, mapping every single candidate to:
* **Recommended Editorial Action** (tailored to its `reason_code` and `word_count`)
* **Reason Code** (`STALE_STRIKING_DISTANCE` or `HIGH_VISIBILITY_AGING`)
* **Confidence Note** (based on prior impression volume and signal completeness)
* **What Would Make It Wrong** (specific failure mode to verify before rewriting)

Across the top 20 picks:
* **Ranks #1–#12 (`STALE_STRIKING_DISTANCE`):** High-impression pages sitting around positions `4`–`12` that have not been updated in `300+` days. **Action:** Refresh outdated facts/stats, update publish/modified dates, and align headings with current search intent. **Confidence:** High when `impressions_prev_30d >= 5,000` and `word_count` is present. **What would make it wrong:** The topic is inherently evergreen (e.g., a timeless definition or historical event) where age does not cause decay, or a recent traffic drop was seasonal rather than quality-driven.
* **Ranks #13–#20 (`HIGH_VISIBILITY_AGING` / `STALE_STRIKING_DISTANCE`):** High-exposure pages with strong historical impressions (`impressions_prev_30d >= 2,000`) and >180 days since update. **Action:** Audit top-3 competing SERP results and expand missing subtopics or thin sections. **Confidence:** Medium-High; slightly lower when `word_count` is missing (`NaN`) or `avg_position` is near the top-3 ceiling (`< 4.0`), where edits risk disrupting an already top-ranking page. **What would make it wrong:** The page ranks #1–#3 for branded/navigational queries where an aggressive rewrite could hurt existing CTR.

In [4]:
top20 = ranked_queue.head(20).copy()

def build_audit_fields(row: pd.Series) -> pd.Series:
    rc = row["reason_code"]
    imp = row["impressions_prev_30d"]
    pos = row["avg_position"]
    wc = row["word_count"]

    if rc == "STALE_STRIKING_DISTANCE":
        action = "Refresh outdated sections, update stats/examples, and optimize H2s for page-1 lift"
    elif rc == "HIGH_VISIBILITY_AGING":
        action = "Audit SERP competitors and expand thin or aging sections to protect high impression volume"
    else:
        action = "Tighten title/meta snippet and internal links to improve CTR"

    if imp >= 5000 and pd.notna(wc):
        conf = "High (strong baseline volume + complete metadata)"
    elif imp >= 1000:
        conf = "Medium-High (solid impression volume; verify word count in CMS)"
    else:
        conf = "Medium (moderate impression volume; check trend stability)"

    if pos < 5:
        risk = "Already top-5 rank; aggressive rewriting could disrupt stable rankings"
    elif pd.isna(wc):
        risk = "Missing word_count; page could be a non-article landing/tool page not suited for blog refresh"
    else:
        risk = "Seasonal query spike or evergreen topic where age does not equal content decay"

    return pd.Series([action, conf, risk], index=["recommended_action", "confidence_note", "what_would_make_it_wrong"])

top20[["recommended_action", "confidence_note", "what_would_make_it_wrong"]] = top20.apply(build_audit_fields, axis=1)

review_cols = [
    "priority_rank",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "recommended_action",
    "confidence_note",
    "what_would_make_it_wrong",
]
pd.set_option("display.max_colwidth", 90)
display(top20[review_cols])


,priority_rank,content_id,baseline_action_score,reason_code,recommended_action,confidence_note,what_would_make_it_wrong
0,1,content_72496874f806,74.67,STALE_STRIKING_DISTANCE,"Refresh outdated sections, update stats/examples, and optimize H2s for page-1 lift",Medium (moderate impression volume; check trend stability),Seasonal query spike or evergreen topic where age does not equal content decay
1,2,content_02b0d6e30129,74.55,STALE_STRIKING_DISTANCE,"Refresh outdated sections, update stats/examples, and optimize H2s for page-1 lift",Medium (moderate impression volume; check trend stability),Missing word_count; page could be a non-article landing/tool page not suited for blog ...
2,3,content_7f116ae1f6f5,71.09,STALE_STRIKING_DISTANCE,"Refresh outdated sections, update stats/examples, and optimize H2s for page-1 lift",Medium (moderate impression volume; check trend stability),Seasonal query spike or evergreen topic where age does not equal content decay
3,4,content_5fe46e04994d,71.05,STRIKING_DISTANCE_OPTIMIZE,Tighten title/meta snippet and internal links to improve CTR,Medium-High (solid impression volume; verify word count in CMS),Already top-5 rank; aggressive rewriting could disrupt stable rankings
4,5,content_2c2606c5d176,70.46,STRIKING_DISTANCE_OPTIMIZE,Tighten title/meta snippet and internal links to improve CTR,Medium-High (solid impression volume; verify word count in CMS),Already top-5 rank; aggressive rewriting could disrupt stable rankings
5,6,content_01908772c6db,68.96,STRIKING_DISTANCE_OPTIMIZE,Tighten title/meta snippet and internal links to improve CTR,Medium-High (solid impression volume; verify word count in CMS),Already top-5 rank; aggressive rewriting could disrupt stable rankings
6,7,content_3d94572c3a35,68.78,STRIKING_DISTANCE_OPTIMIZE,Tighten title/meta snippet and internal links to improve CTR,High (strong baseline volume + complete metadata),Already top-5 rank; aggressive rewriting could disrupt stable rankings
7,8,content_bb5bd5f771dc,68.35,STRIKING_DISTANCE_OPTIMIZE,Tighten title/meta snippet and internal links to improve CTR,Medium-High (solid impression volume; verify word count in CMS),Already top-5 rank; aggressive rewriting could disrupt stable rankings
8,9,content_89fcb6f35525,68.11,STRIKING_DISTANCE_OPTIMIZE,Tighten title/meta snippet and internal links to improve CTR,High (strong baseline volume + complete metadata),Already top-5 rank; aggressive rewriting could disrupt stable rankings
9,10,content_d296f221f23b,67.68,STRIKING_DISTANCE_OPTIMIZE,Tighten title/meta snippet and internal links to improve CTR,Medium-High (solid impression volume; verify word count in CMS),Already top-5 rank; aggressive rewriting could disrupt stable rankings


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks & Strict Leakage Audit

#### 1. Which Picks Look Weak and Why?
* **Pages with Missing `word_count` (`NaN`):** Some top-scored pages lack `word_count` telemetry. Without knowing if a URL is a 3,000-word pillar guide or a 100-word category/index page, a rule based purely on staleness + impressions can misclassify non-editorial pages as "stale articles."
* **Single-Client Concentration Risk:** If one large `client_id` dominates high-impression aged pages, a global uncalibrated rule can flood the top-20 queue with pages from just one or two sites rather than balancing across clients.
* **Evergreen High-Staleness False Positives:** Pages capped at the 365-day staleness maximum (`days_since_last_update >= 365`) receive full staleness points even when their underlying topic is evergreen and their clicks are steady.

#### 2. Leakage Audit Confirmation
* **Zero Future Windows Used:** Our rule uses **only** `impressions_prev_30d`, `avg_position`, `days_since_last_update`, and `content_age_days`. None of the forward outcome columns (`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `trend_direction`, or `trend_pct`) were touched during scoring or ranking.
* **Zero Product / Internal Outcome Flags Used:** No downstream label proxies or internal flags were included in `compute_baseline_score` or `assign_reason_code`.

In [5]:
# 1. Inspect potential weak picks in the Top 20 (e.g., missing word_count or single-client concentration)
missing_wc_top20 = top20["word_count"].isna().sum()
unique_clients_top20 = top20["client_id"].nunique()
print(f"Top-20 Weak-Pick Audit:")
print(f" - Picks with missing word_count (possible non-article/unmeasured pages): {missing_wc_top20} / 20")
print(f" - Distinct clients represented in Top 20: {unique_clients_top20} / 20")

# Show any top-20 rows with missing word_count
weak_rows = top20[top20["word_count"].isna()][["priority_rank", "content_id", "client_id", "baseline_action_score", "impressions_prev_30d"]]
if len(weak_rows) > 0:
    print("\nTop-20 rows flagged for missing word_count:")
    display(weak_rows)

# 2. Programmatic Leakage Check: assert zero future/label columns exist in the exported queue
forbidden_leaky_cols = {
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "trend_direction",
    "trend_pct",
    "needs_refresh_proxy",
}
exported_cols = set(pd.read_csv("work/outputs/baseline_action_score.csv", nrows=1).columns)
leaked_found = exported_cols.intersection(forbidden_leaky_cols)

assert len(leaked_found) == 0, f"LEAKAGE DETECTED: {leaked_found}"
print(f"\n[LEAKAGE CHECK PASSED] Exported columns: {sorted(exported_cols)}")
print("Confirmed: 0 future-window columns and 0 product/target flags used.")


Top-20 Weak-Pick Audit:
 - Picks with missing word_count (possible non-article/unmeasured pages): 11 / 20
 - Distinct clients represented in Top 20: 5 / 20

Top-20 rows flagged for missing word_count:


,priority_rank,content_id,client_id,baseline_action_score,impressions_prev_30d
1,2,content_02b0d6e30129,client_19581e27de,74.55,159
3,4,content_5fe46e04994d,client_4e07408562,71.05,218786
4,5,content_2c2606c5d176,client_19581e27de,70.46,164079
5,6,content_01908772c6db,client_19581e27de,68.96,66092
7,8,content_bb5bd5f771dc,client_19581e27de,68.35,63306
9,10,content_d296f221f23b,client_19581e27de,67.68,49658
10,11,content_3f403bb36296,client_19581e27de,67.63,52672
16,17,content_4a18c07e5357,client_4e07408562,67.26,43906
17,18,content_bc104ff59d1a,client_19581e27de,66.99,45630
18,19,content_cd36fc9f75ce,client_19581e27de,66.99,29779



[LEAKAGE CHECK PASSED] Exported columns: ['avg_position', 'baseline_action_score', 'client_id', 'content_id', 'days_since_last_update', 'impressions_prev_30d', 'priority_rank', 'reason_code', 'word_count']
Confirmed: 0 future-window columns and 0 product/target flags used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check
Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.